# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 **My Rule:**<br> If CTR is low, engagement is low, low scroll event rate, position is greater than or equal to 10 and page is not updated for a quite long time than refresh page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed
""")


Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

<h2>2.1: Import Libraries</h2>

In [55]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
import duckdb

<h2>2.2: Get data from hugging face Flyrank repo</h2>

In [56]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h2>2.3: Taken sample 60day window data from fact_daily</h2>

In [57]:
clients_last_3m = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_pageviews, ga4_sessions, ga4_engaged_sessions, scroll_events
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-05-30'
      AND client_has_gsc == 'true'
      AND client_has_ga4 == 'true'
      AND gsc_data_available == 'true'
      AND ga4_data_available == 'true'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<h2></h2>

<h2>Checking length</h2>

In [58]:
print(len(clients_last_3m))

1111616


<h2>Check columns</h2>

In [59]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'],
      dtype='object')

<h2>Check NULL impressions if any</h2>

In [60]:
clients_last_3m["gsc_impressions"].isna().sum()

np.int64(0)

<h2>2.4: Load dim_content table data in pandas dataframe</h2>

In [61]:
dimf_content = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_updated_date
    FROM {TABLES['dim_content']}
""").df()

<h2>2.5: Calculate CTR, Engagement rate, Scroll rate, Days since update</h2>

In [63]:
reference_date = dimf_content["content_updated_date"].max()
clients_last_3m["ctr"] = clients_last_3m["gsc_clicks"] / clients_last_3m["gsc_impressions"]
clients_last_3m["engagement_rate"] = clients_last_3m["ga4_engaged_sessions"] / clients_last_3m["ga4_sessions"]
clients_last_3m["scroll_rate"] = clients_last_3m["scroll_events"] / clients_last_3m["ga4_pageviews"].replace(0, pd.NA)
clients_last_3m = clients_last_3m.merge(
    dimf_content,
    on=("client_hash_id", "content_hash_id"),
    how="left"
)
clients_last_3m["days_since_update"] = (reference_date - clients_last_3m["content_updated_date"]).dt.days

<h2>Checking columns</h2>

In [64]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events', 'ctr', 'engagement_rate',
       'scroll_rate', 'content_updated_date', 'days_since_update'],
      dtype='object')

<h2>2.6: Drop unnecessary columns</h2>

In [65]:
clients_last_3m = clients_last_3m.drop(columns=['content_updated_date', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'])

<h2>Checking columns</h2>

In [66]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update'],
      dtype='object')

<h2>2.7:
2 Signals: <br>  
1. CTR-vs-position (FlyRank signal)<br>
2. Engagement rate vs Scroll rate </h2>

In [67]:
# SIGNAL 1:
clients_last_3m["position_bucket"] = pd.cut(
    clients_last_3m["gsc_avg_position"],
    bins=[0, 3, 10, 20, 100, 100000],
    labels=["1-3", "4-10", "11-20", "21-100", "100+"]
)

signal1 = clients_last_3m.groupby("position_bucket").agg(
    avg_ctr=("ctr", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 1: CTR-vs-position ===")
print(signal1)
print("Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr")

# SIGNAL 2:
clients_last_3m["engagement_bucket"] = pd.cut(
    clients_last_3m["engagement_rate"],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=["low", "medium", "high", "very_high"]
)

signal2 = clients_last_3m.groupby("engagement_bucket").agg(
    avg_scroll_rate=("scroll_rate", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 2: Engagement rate vs Scroll rate ===")
print(signal2)
print("Verdict: CONFIRMED if engagement is high the average scroll rate will also be high")

/tmp/ipykernel_2715/1362672175.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = clients_last_3m.groupby("position_bucket").agg(
/tmp/ipykernel_2715/1362672175.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = clients_last_3m.groupby("engagement_bucket").agg(


=== Signal 1: CTR-vs-position ===
                  avg_ctr       n
position_bucket                  
1-3              0.028330   66002
4-10             0.013217  530906
11-20            0.011634  233780
21-100           0.007033  274790
100+             0.038492     248
Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr
=== Signal 2: Engagement rate vs Scroll rate ===
                  avg_scroll_rate      n
engagement_bucket                       
low                       0.15123  25246
medium                   0.368508  23512
high                     0.539958    782
very_high                0.804206  27282
Verdict: CONFIRMED if engagement is high the average scroll rate will also be high


<h2>2.8: Check details of columns = ctr, engagement_rate, scroll_rate, days_since_update to find threshold values</h2>

In [68]:
print(clients_last_3m[clients_last_3m["ctr"] < 0.000005])

        report_date           client_hash_id           content_hash_id  \
0        2026-04-01  client_9958f0a7ae1df715  content_810cf06597918291   
2        2026-04-01  client_9958f0a7ae1df715  content_c4002ce386c98905   
5        2026-04-01  client_9958f0a7ae1df715  content_dc2c2198d9631650   
7        2026-04-01  client_9958f0a7ae1df715  content_28d211a926b33519   
8        2026-04-01  client_9958f0a7ae1df715  content_01abeb8b40591eec   
...             ...                      ...                       ...   
1111606  2026-05-30  client_1a8bf67cad4ee525  content_3811343b165eb63a   
1111609  2026-05-30  client_1a8bf67cad4ee525  content_fadf7ae978fd082e   
1111612  2026-05-30  client_1a8bf67cad4ee525  content_5eb9c0b1de0202d2   
1111613  2026-05-30  client_1a8bf67cad4ee525  content_349c92d5cd468777   
1111614  2026-05-30  client_1a8bf67cad4ee525  content_978d1979c92d5bdb   

         gsc_impressions  gsc_avg_position  ctr  engagement_rate scroll_rate  \
0                      1       

In [69]:
clients_last_3m["engagement_rate"].value_counts()

,count
engagement_rate,
0.000000,1017510
1.000000,27265
0.500000,14167
0.333333,8316
0.250000,5559
...,...
0.041237,1
0.074468,1
0.065421,1


In [70]:
clients_last_3m["scroll_rate"].value_counts()

,count
scroll_rate,
0.0,927122
1.0,59581
0.5,42139
0.25,14407
0.333333,14260
...,...
0.348837,1
0.082873,1
0.019504,1


In [71]:
clients_last_3m["days_since_update"].value_counts()

,count
days_since_update,
47,269663
131,169000
25,78984
49,77713
14,48275
...,...
255,2
400,2
104,2


<h2>2.9: Threshold values</h2>

In [79]:
ctr_threshold = 0.00005
engagement_threshold = 0
scroll_threshold = 0
staleness_threshold = 30

<h2>2.10: Calculate decline_score proxy label</h2>

In [80]:
clients_last_3m["decline_score"] = (
    (clients_last_3m["ctr"] < ctr_threshold).astype(int) +
    (clients_last_3m["engagement_rate"] <= engagement_threshold).astype(int) +
    (clients_last_3m["scroll_rate"] <= scroll_threshold).astype(int) +
    (clients_last_3m["gsc_avg_position"] >= 10).astype(int) +
    (clients_last_3m["days_since_update"] > staleness_threshold).astype(int)
)

<h2>2.11: Sort decline_score proxy label in descending order for data window</h2>

In [81]:
ranked_queue = clients_last_3m.sort_values("decline_score", ascending=False)

<h2>2.12: Print top 10 ranked rows</h2>

In [82]:
print("Top 10")
ranked_queue[:10]

Top 10


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
7,2026-04-01,client_9958f0a7ae1df715,content_28d211a926b33519,1,27.000000,0.0,0.0,0.0,38,21-100,NaN,5
1111598,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
8,2026-04-01,client_9958f0a7ae1df715,content_01abeb8b40591eec,15,15.133333,0.0,0.0,0.0,38,11-20,NaN,5
467084,2026-05-02,client_73cda7b4e4f265ea,content_c2366b6258fa1f71,13,11.615385,0.0,0.0,0.0,131,11-20,NaN,5
467092,2026-05-02,client_73cda7b4e4f265ea,content_ccd341b33d81ace7,54,13.537037,0.0,0.0,0.0,47,11-20,NaN,5
1111575,2026-05-30,client_1a8bf67cad4ee525,content_c42f8b32cb6fefdc,11,10.545455,0.0,0.0,0.0,47,11-20,NaN,5
32,2026-04-01,client_9958f0a7ae1df715,content_3c3b575d53a71932,28,19.464286,0.0,0.0,0.0,38,11-20,NaN,5
33,2026-04-01,client_9958f0a7ae1df715,content_e819a76f8434a765,6,32.500000,0.0,0.0,0.0,38,21-100,NaN,5
34,2026-04-01,client_9958f0a7ae1df715,content_d8288f22519f7d53,12,26.250000,0.0,0.0,0.0,38,21-100,NaN,5
37,2026-04-01,client_9958f0a7ae1df715,content_c040204537f06a59,67,31.955224,0.0,0.0,0.0,38,21-100,NaN,5


<h2>2.13: Save ranked queue data in 'CSV' format in 'work/outputs' directory and locak machine</h2>

In [22]:
from google.colab import files

# Ensure the output directory exists
#os.makedirs('work/outputs', exist_ok=True)

# Save the ranked queue to CSV
#output_path = 'work/outputs/baseline_action_score.csv'
#ranked_queue.to_csv(output_path, index=False)

#print(f'File saved to {output_path}. Starting download...')

# Trigger browser download to local machine
#files.download(output_path)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

<h2>3.1: Top 20</h2>

In [83]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue[:20]

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
7,2026-04-01,client_9958f0a7ae1df715,content_28d211a926b33519,1,27.000000,0.0,0.0,0.0,38,21-100,NaN,5
1111598,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
8,2026-04-01,client_9958f0a7ae1df715,content_01abeb8b40591eec,15,15.133333,0.0,0.0,0.0,38,11-20,NaN,5
467084,2026-05-02,client_73cda7b4e4f265ea,content_c2366b6258fa1f71,13,11.615385,0.0,0.0,0.0,131,11-20,NaN,5
467092,2026-05-02,client_73cda7b4e4f265ea,content_ccd341b33d81ace7,54,13.537037,0.0,0.0,0.0,47,11-20,NaN,5
1111575,2026-05-30,client_1a8bf67cad4ee525,content_c42f8b32cb6fefdc,11,10.545455,0.0,0.0,0.0,47,11-20,NaN,5
32,2026-04-01,client_9958f0a7ae1df715,content_3c3b575d53a71932,28,19.464286,0.0,0.0,0.0,38,11-20,NaN,5
33,2026-04-01,client_9958f0a7ae1df715,content_e819a76f8434a765,6,32.500000,0.0,0.0,0.0,38,21-100,NaN,5
34,2026-04-01,client_9958f0a7ae1df715,content_d8288f22519f7d53,12,26.250000,0.0,0.0,0.0,38,21-100,NaN,5
37,2026-04-01,client_9958f0a7ae1df715,content_c040204537f06a59,67,31.955224,0.0,0.0,0.0,38,21-100,NaN,5


<h2>3.2: Action: "Refresh_needed" because decline score is highest </h2>

<h2>3.3: Reason code: <br>
All signals weak for all 20 rows</h2>

<h2>3.4: Confidence note:</h2>
1. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. And impression is just 1.<br>
2. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are somewhat there<br>
3. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are somewhat there.<br>
4. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 100. <br>
5. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are 54 which are somewhat reasonable for this confidence.<br>
6. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies around 10 bucket. The day_since_update is also > 30. Because of where it lies in position medium confidence.<br>
7.  Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. The confidence is this because impressions are somewhat reasonable. <br>
8. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are also very less. <br>
9. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are also very less.<br>
10. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are 67 and so the above 0 values are somewhat problamatic.<br>
11. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low as well as position.<br>
12. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and avg. position is fine but low impressions.<br>
13. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low as well as position.<br>
14. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and avg. position is lower but somewhat reasonable impressions.<br>
15. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low and position is lower also.<br>
16. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low but position is close to 10 but due to other strong factors high confidence.<br>
17. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
18. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
19. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
20. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.


<h2>3.5: What would make it wrong?</h2>
1. Would be wrong if this page simply hasn't accumulated enough traffic yet to judge — not a real performance problem.<br>
2. Would be wrong if the page is on the edge of page 1 and just needs time, not a content refresh.<br>
3. Would be wrong if 38 days is too soon to call this "long neglected" — may just be normal maturation lag.<br>
4. Would be wrong if this page's low traffic is due to a niche/low-demand topic rather than declining quality.<br>
5. Would be wrong if this traffic came from a single anomalous spike (e.g., bot traffic or a one-off referral) rather than sustained real visits.<br>
6. Would be wrong if position is measurement noise right at the 10/11 boundary, making the "position ≥ 10" trigger arbitrary here.<br>
7. Would be wrong if 28 impressions with genuinely 0 engagement reflects a mismatched search intent, not something a refresh would fix.<br>
8. Would be wrong if this page is simply new/low-priority and not worth refresh effort yet.<br>
9. Would be wrong if low volume reflects a legitimately low-demand keyword, not decaying content.<br>
10. Would be wrong if impressions are inflated by irrelevant/broad-match queries that were never going to convert regardless of content quality.<br>
11. Would be wrong if this is simply a poor keyword-content match rather than a staleness issue.<br>
12. Would be wrong if the page is too new for its traffic to have stabilized.<br>
13. Would be wrong if low volume reflects niche topic demand, not quality decay.<br>
14. Would be wrong if this page's topic naturally has zero-click search behavior (e.g., answers fully shown in the snippet).<br>
15. Would be wrong if this page gets almost no search demand regardless of freshness.<br>
16. Would be wrong if the ranking position itself is unstable/fluctuating day to day, making this single snapshot misleading.<br>
17. Would be wrong if low impressions reflect seasonal/cyclical demand rather than a real decline.<br>
18. Would be wrong if position (10.4) is borderline enough that this page is essentially performing fine.<br>
19. Would be wrong if this reflects low search demand for the topic, not neglect.<br>
20. Would be wrong if position (10.4) is right at the boundary and effectively equivalent to a "good" ranking.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue["gsc_impressions"].describe()

,gsc_impressions
count,1.111616e+06
mean,2.223129e+02
std,5.134163e+02
min,1.000000e+00
25%,2.300000e+01
50%,7.900000e+01
75%,2.270000e+02
max,8.231700e+04


In [40]:
ranked_queue["scroll_rate"].value_counts()

,count
scroll_rate,
0.0,927122
1.0,59581
0.5,42139
0.25,14407
0.333333,14260
...,...
0.225806,1
0.026786,1
0.246575,1


In [42]:
ranked_queue["ctr"].value_counts().head(20)

,count
ctr,
0.000000,559374
0.032258,2604
0.045455,2593
0.041667,2507
0.050000,2500
0.058824,2478
0.033333,2467
0.035714,2464
0.052632,2463


In [77]:
print((ranked_queue["days_since_update"] < 0).sum())
print((ranked_queue["days_since_update"] > 0).sum())
print(ranked_queue["days_since_update"].sum())


0
1097276
49995494


In [78]:
print(f"reference_date being used: {reference_date}")
print(dimf_content["content_updated_date"].describe())
print(dimf_content["content_updated_date"].max())

reference_date being used: 2026-07-06 00:00:00
count                        519606
mean     2026-02-24 20:26:49.089964
min             2024-10-28 00:00:00
25%             2026-02-25 00:00:00
50%             2026-05-20 00:00:00
75%             2026-06-01 00:00:00
max             2026-07-06 00:00:00
Name: content_updated_date, dtype: object
2026-07-06 00:00:00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.